# Config


In [1]:
import os 

#Move path to another directory
newpath = r'C:\Users\alonso.valderrama.ex\OneDrive - ACLARA\Desktop\Magister\repos\LibEER\LibEER'
os.chdir(newpath)
#Print current working directory
print(os.getcwd())

C:\Users\alonso.valderrama.ex\OneDrive - ACLARA\Desktop\Magister\repos\LibEER\LibEER


# Run code DGCNN

In [ ]:
#python DGCNN_train.py -metrics 'acc' 'macro-f1' -metric_choose 'macro-f1' -setting seed_sub_dependent_train_val_test_setting -dataset_path /data1/cxx/SEED数据集/SEED/ -dataset seed_de_lds -batch_size 32 -seed 2024 -epochs 80 -lr 0.0015 -onehot >DGCNN/b32_lr0.0015.log


import numpy as np

from models.Models import Model
from config.setting import seed_sub_dependent_front_back_setting, preset_setting, set_setting_by_args
from data_utils.load_data import get_data
from data_utils.split import merge_to_part, index_to_data, get_split_index
from utils.args import get_args_parser
from utils.store import make_output_dir
from utils.utils import state_log, result_log, setup_seed, sub_result_log
from Trainer.training import train
from models.DGCNN import NewSparseL2Regularization
import torch
import torch.optim as optim
import torch.nn as nn

def main(args):
    if args.setting is not None:
        setting = preset_setting[args.setting](args)
    else:
        setting = set_setting_by_args(args)
    setup_seed(args.seed)
    data, label, channels, feature_dim, num_classes = get_data(setting)
    data, label = merge_to_part(data, label, setting)
    device = torch.device(args.device)
    best_metrics = []
    dependent_metrics = [[] for _ in range(len(data))]
    for rridx, (data_i, label_i) in enumerate(zip(data, label), 1):
        tts = get_split_index(data_i, label_i, setting)
        for ridx, (train_indexes, test_indexes, val_indexes) in enumerate(zip(tts['train'], tts['test'], tts['val']), 1):
            setup_seed(args.seed)
            if val_indexes[0] == -1:
                print(f"train indexes:{train_indexes}, test indexes:{test_indexes}")
            else:
                print(f"train indexes:{train_indexes}, val indexes:{val_indexes}, test indexes:{test_indexes}")

            test_sub_label = None

            # record who each sample belong to
            if setting.experiment_mode == "subject-independent":
                # extract the subject label
                train_data, train_label, val_data, val_label, test_data, test_label = \
                    index_to_data(data_i, label_i, train_indexes, test_indexes, val_indexes, True)
                test_sub_num = len(test_data)
                test_sub_label = []
                for i in range(test_sub_num):
                    test_sub_count = len(test_data[i])
                    test_sub_label.extend([i+1 for j in range(test_sub_count)])
                test_sub_label = np.array(test_sub_label)

            # split train and test data by specified experiment mode
            train_data, train_label, val_data, val_label, test_data, test_label = \
                index_to_data(data_i, label_i, train_indexes, test_indexes, val_indexes, args.keep_dim)

            # model to train
            if len(val_data) == 0:
                val_data = test_data
                val_label = test_label
            model = Model['DGCNN'](channels, feature_dim, num_classes)
            # Train one round using the train one round function defined in the model
            dataset_train = torch.utils.data.TensorDataset(torch.Tensor(train_data), torch.Tensor(train_label))
            dataset_val = torch.utils.data.TensorDataset(torch.Tensor(val_data), torch.Tensor(val_label))
            dataset_test = torch.utils.data.TensorDataset(torch.Tensor(test_data), torch.Tensor(test_label))
            optimizer = optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4, eps=1e-4)
            criterion = nn.CrossEntropyLoss()
            loss_func = NewSparseL2Regularization(0.01).to(device)
            output_dir = make_output_dir(args, "DGCNN")
            round_metric = train(model=model, dataset_train=dataset_train, dataset_val=dataset_val, dataset_test=dataset_test, device=device,
                                 output_dir=output_dir, metrics=args.metrics, metric_choose=args.metric_choose, optimizer=optimizer,
                                 batch_size=args.batch_size, epochs=args.epochs, criterion=criterion, loss_func=loss_func, loss_param=model) #test_sub_label=test_sub_label,
            # if sub indep, then return
            best_metrics.append(round_metric)
            if setting.experiment_mode == "subject-dependent":
                dependent_metrics[rridx - 1].append(round_metric)


    # best metrics: every round metrics dict
    # subjects dependent metrics: (subject, sub_round_metric)
    # subjects independent metrics: (round_metric{mean:"",std:""})
    if setting.experiment_mode == "subject-dependent":
        sub_result_log(args, dependent_metrics)
        print(dependent_metrics)
    else:
        result_log(args, best_metrics)
        print(best_metrics)




In [ ]:
from utils.args import get_args_parser
from DGCNN_train import main

parser = get_args_parser()

args = parser.parse_args([])

# Setear manualmente
args.metrics = ['acc', 'macro-f1']
args.metric_choose = 'macro-f1'
args.setting = 'seed_sub_dependent_train_val_test_setting'
args.dataset_path = r'C:\Users\alonso.valderrama.ex\OneDrive - ACLARA\Desktop\Magister\data\SEED'
args.dataset = 'seed_de_lds'
args.batch_size = 32
args.seed = 2024
args.epochs = 80
args.lr = 0.0015
args.onehot = True

# Choose output locations
args.output_dir = r'C:\Users\alonso.valderrama.ex\OneDrive - ACLARA\Desktop\Magister\outputs'   # where model checkpoints go
args.log_dir = r'C:\Users\alonso.valderrama.ex\OneDrive - ACLARA\Desktop\Magister\outputs'             # where result logs go

main(args)

# DGCNN Escolares

In [ ]:
#python DGCNN_train.py -metrics 'acc' 'macro-f1' -metric_choose 'macro-f1' -setting seed_sub_dependent_train_val_test_setting -dataset_path /data1/cxx/SEED数据集/SEED/ -dataset seed_de_lds -batch_size 32 -seed 2024 -epochs 80 -lr 0.0015 -onehot >DGCNN/b32_lr0.0015.log


import numpy as np

from models.Models import Model
from config.setting import seed_sub_dependent_front_back_setting, preset_setting, set_setting_by_args
from data_utils.load_data import get_data
from data_utils.split import merge_to_part, index_to_data, get_split_index
from utils.args import get_args_parser
from utils.store import make_output_dir
from utils.utils import state_log, result_log, setup_seed, sub_result_log
from Trainer.training import train
from models.DGCNN import NewSparseL2Regularization
import torch
import torch.optim as optim
import torch.nn as nn

def main(args):
    if args.setting is not None:
        setting = preset_setting[args.setting](args)
    else:
        setting = set_setting_by_args(args)
    setup_seed(args.seed)
    data, label, channels, feature_dim, num_classes = get_data(setting)
    data, label = merge_to_part(data, label, setting)
    device = torch.device(args.device)
    best_metrics = []
    dependent_metrics = [[] for _ in range(len(data))]
    for rridx, (data_i, label_i) in enumerate(zip(data, label), 1):
        tts = get_split_index(data_i, label_i, setting)
        for ridx, (train_indexes, test_indexes, val_indexes) in enumerate(zip(tts['train'], tts['test'], tts['val']), 1):
            setup_seed(args.seed)
            if val_indexes[0] == -1:
                print(f"train indexes:{train_indexes}, test indexes:{test_indexes}")
            else:
                print(f"train indexes:{train_indexes}, val indexes:{val_indexes}, test indexes:{test_indexes}")

            test_sub_label = None

            # record who each sample belong to
            if setting.experiment_mode == "subject-independent":
                # extract the subject label
                train_data, train_label, val_data, val_label, test_data, test_label = \
                    index_to_data(data_i, label_i, train_indexes, test_indexes, val_indexes, True)
                test_sub_num = len(test_data)
                test_sub_label = []
                for i in range(test_sub_num):
                    test_sub_count = len(test_data[i])
                    test_sub_label.extend([i+1 for j in range(test_sub_count)])
                test_sub_label = np.array(test_sub_label)

            # split train and test data by specified experiment mode
            train_data, train_label, val_data, val_label, test_data, test_label = \
                index_to_data(data_i, label_i, train_indexes, test_indexes, val_indexes, args.keep_dim)

            # model to train
            if len(val_data) == 0:
                val_data = test_data
                val_label = test_label
            model = Model['DGCNN'](channels, feature_dim, num_classes)
            # Train one round using the train one round function defined in the model
            dataset_train = torch.utils.data.TensorDataset(torch.Tensor(train_data), torch.Tensor(train_label))
            dataset_val = torch.utils.data.TensorDataset(torch.Tensor(val_data), torch.Tensor(val_label))
            dataset_test = torch.utils.data.TensorDataset(torch.Tensor(test_data), torch.Tensor(test_label))
            optimizer = optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4, eps=1e-4)
            criterion = nn.CrossEntropyLoss()
            loss_func = NewSparseL2Regularization(0.01).to(device)
            output_dir = make_output_dir(args, "DGCNN")
            round_metric = train(model=model, dataset_train=dataset_train, dataset_val=dataset_val, dataset_test=dataset_test, device=device,
                                 output_dir=output_dir, metrics=args.metrics, metric_choose=args.metric_choose, optimizer=optimizer,
                                 batch_size=args.batch_size, epochs=args.epochs, criterion=criterion, loss_func=loss_func, loss_param=model) #test_sub_label=test_sub_label,
            # if sub indep, then return
            best_metrics.append(round_metric)
            if setting.experiment_mode == "subject-dependent":
                dependent_metrics[rridx - 1].append(round_metric)


    # best metrics: every round metrics dict
    # subjects dependent metrics: (subject, sub_round_metric)
    # subjects independent metrics: (round_metric{mean:"",std:""})
    if setting.experiment_mode == "subject-dependent":
        sub_result_log(args, dependent_metrics)
        print(dependent_metrics)
    else:
        result_log(args, best_metrics)
        print(best_metrics)

